# Model probe: does a trained model evaluate high reward as high reward, and does it propose impossible moves?

Evaluation only. Everything here uses the real maze and the exact DP; nothing feeds training.

**Reward prediction.** Synthetic trajectories with known outcomes from every start cell: a shortest path
(`optimal`), shortest paths with round trips inserted (`detour<k>`, arriving `2k` steps late), `w` random steps
then a shortest path (`wander<w>`), and plain random walks. The NOR value head is read at every prefix and
compared with the achieved bin and with the exact posterior `h[t, s_t]` from the DP.

**Impossible moves.** The policy's mass on actions that hit a wall, for NOR and for each conditioned bin,
against the exact conditioned random walk's wall mass at the same prefix; and the dynamics head's mass off the
real next cell.


In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}
RUN = "late_threshold/mc_all_s0"  #@param {type:"string"}
MAZE_KW = dict(binning="geometric", n_bins=24)   # must match the run (its K is checked)
COND = "threshold"                                # must match the run: "bin" or "threshold" (stored in its params.pkl)


In [ ]:
# Clone (or update) the repo and make sure the dependencies are importable.
import os, subprocess, sys

url = REPO
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret: public repo

if not os.path.exists("/content/sillyrl/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
else:
    subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
os.chdir("/content/sillyrl")
sys.path.insert(0, "/content/sillyrl")

# Colab's preinstalled flax can lag its JAX (e.g. flax calling jax.core APIs that JAX 0.11 removed), so always
# upgrade flax and optax before importing them. pip keeps Colab's JAX if it already satisfies them; if it had
# to upgrade JAX, bring the GPU plugin (jax-cuda*) to the same version so the runtime doesn't fall back to CPU.
import importlib.metadata as md
jax_before = md.version("jax")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
jax_after = md.version("jax")
if jax_after != jax_before:
    plugins = sorted({d.metadata["Name"] for d in md.distributions()
                      if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
    if plugins:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
    print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices())

In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])

In [ ]:
import os, pickle, numpy as np, jax.numpy as jnp
import matplotlib.pyplot as plt
from maze_consistency.dataset import load as load_data
from maze_consistency.tokens import Tokenizer
from maze_consistency.model import ModelConfig, MazeTransformer, make_forward
from maze_consistency.dp import compute_ground_truth
from maze_consistency.train import load_run
from maze_consistency import probe as P

import subprocess, sys
if not os.path.exists("data/canonical/rollouts.npz"):          # the dataset is built, not stored in the repo
    subprocess.run([sys.executable, "run.py", "dataset"], check=True)
MAZE, DATA = load_data(**MAZE_KW)
TOK = Tokenizer(MAZE, cond=COND)
GT = compute_ground_truth(MAZE)
params, cfg = load_run(RUN)
import pickle as _pk
_meta = _pk.load(open(os.path.join(os.environ.get('RUNS_DIR', 'runs'), RUN, 'params.pkl'), 'rb'))
if 'cond' in _meta and _meta['cond'] != COND:
    raise ValueError(f"run was trained with cond={_meta['cond']!r}; set COND to match")
assert cfg.K == MAZE.K, f"run has K={cfg.K} but MAZE_KW gives K={MAZE.K}; set MAZE_KW to the run's binning"
MODEL = MazeTransformer(cfg); FWD = make_forward(MODEL, TOK)
print(RUN, "|", cfg, "|", MAZE)
LIVE = [k for k in range(MAZE.K) if k not in MAZE.empty_bins]
_best_at = lambda dd: int(MAZE.best_bin(np.flatnonzero(MAZE.dist == dd)[0]))
FAR_BINS = sorted({_best_at(20), _best_at(15), _best_at(10)})
print("usable bins:", LIVE, "| far starts' best bins:", FAR_BINS)


## 1. Synthetic trajectories

In [ ]:
TRAJ = P.synthetic_trajectories(MAZE, seed=0, n_per_start=2, detours=(2, 5, 10), wander=(5, 15, 30), n_random=2)
kinds = sorted(set(TRAJ["kind"]), key=lambda k: (k[:3] != "opt", k))
print(len(TRAJ["length"]), "trajectories from", len(MAZE.start_cells), "starts")
print(f"{'kind':10s} {'n':>5s} {'mean L':>8s} {'reached':>8s} {'mean bin':>9s}")
for k in kinds:
    m = TRAJ["kind"] == k
    print(f"{k:10s} {m.sum():5d} {TRAJ['length'][m].mean():8.1f} {TRAJ['reached'][m].mean():8.2f} {TRAJ['bin'][m].mean():9.2f}")
BUCKET = P.dist_bucket(TRAJ["start_dist"])
BUCKETS = sorted(set(BUCKET), key=lambda b: int(b.split('-')[0]))


## 2. Reward prediction along the trajectory

The question is whether the value head recognises a high-reward trajectory *as it unfolds*. Early in an optimal path from a far start the exact posterior is tiny (the random-walk prior), so the fair comparison there is KL to the exact posterior; late in the path and at the end, the achieved bin should dominate.

In [ ]:
RP = P.reward_prediction(FWD, params, TOK, MAZE, GT, TRAJ)

def table(values, title, fmt="{:6.2f}"):
    print(f"\n{title}\n{'kind':10s}" + "".join(f"{b:>10s}" for b in BUCKETS) + f"{'all':>10s}")
    for k in kinds:
        m = TRAJ["kind"] == k
        cells = [np.nanmean(values[m & (BUCKET == b)]) if (m & (BUCKET == b)).any() else np.nan for b in BUCKETS]
        print(f"{k:10s}" + "".join(f"{fmt.format(c):>10s}" for c in cells) + f"{fmt.format(np.nanmean(values[m])):>10s}")

table(RP["terminal_top"], "terminal prefix: value head argmax == achieved bin   (rows: kind; columns: start distance)")
table(RP["terminal_p"], "terminal prefix: q(achieved bin)")
# at fixed fractions of the trajectory
L = TRAJ["length"].astype(int); N = len(L)
for frac in (0.0, 0.5, 0.9):
    idx = np.round(frac * L).astype(int)
    table(RP["p_bin"][np.arange(N), idx], f"q_t(achieved bin) at t = {frac:.1f} L")
    table(RP["kl"][np.arange(N), np.minimum(idx, np.maximum(L - 1, 0))], f"KL(exact posterior || model) at t = {frac:.1f} L", fmt="{:6.3f}")


In [ ]:
# q_t(achieved bin) as a function of remaining distance, by kind: does belief in the outcome rise as the goal nears?
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k in kinds:
    m = TRAJ["kind"] == k
    rem, p, kl = RP["remaining"][m].ravel(), RP["p_bin"][m].ravel(), RP["kl"][m].ravel()
    rs = np.arange(0, int(np.nanmax(rem)) + 1)
    ax[0].plot(rs, [np.nanmean(p[rem == r]) if (rem == r).any() else np.nan for r in rs], marker=".", label=k)
    ax[1].plot(rs, [np.nanmean(kl[rem == r]) if (rem == r).any() else np.nan for r in rs], marker=".", label=k)
ax[0].set_xlabel("remaining distance to goal"); ax[0].set_ylabel("q_t(achieved bin)"); ax[0].set_yscale("log"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("remaining distance to goal"); ax[1].set_ylabel("KL(exact || model)"); ax[1].set_yscale("log")
for a in ax: a.grid(alpha=.3)
fig.tight_layout(); plt.show()


In [ ]:
# the far-start optimal trajectories in full: model vs exact posterior on the achieved bin at every step
opt_far = np.flatnonzero((TRAJ["kind"] == "optimal") & (TRAJ["start_dist"] >= 15))[:6]
fig, ax = plt.subplots(1, len(opt_far), figsize=(3.2 * len(opt_far), 3.2), squeeze=False)
for a, i in zip(ax[0], opt_far):
    Li, b = int(TRAJ["length"][i]), int(TRAJ["bin"][i])
    ts = np.arange(Li + 1)
    exact = GT.h[ts, TRAJ["positions"][i, :Li + 1].astype(int), b]
    a.plot(ts, RP["p_bin"][i, :Li + 1], marker=".", label="model"); a.plot(ts, exact, ls="--", label="exact")
    a.set_yscale("log"); a.set_title(f"start dist {TRAJ['start_dist'][i]}, bin {b}", fontsize=9); a.set_xlabel("t"); a.grid(alpha=.3)
ax[0, 0].legend(fontsize=7); ax[0, 0].set_ylabel("q_t(achieved bin)")
fig.tight_layout(); plt.show()


## 3. Impossible moves

Wall mass: the policy's probability of an action that hits a wall. NOR should match the uniform walk (0.25 per walled direction). A conditioned high bin should put far less there, as the exact conditioned walk does. `dyn_wrong` is the dynamics head's mass off the real next cell: the hallucination rate that imagined rollouts inherit.

In [ ]:
MODES = (None, *sorted(set(FAR_BINS) | {LIVE[-1]}))
IM = P.impossible_moves(FWD, params, TOK, MAZE, GT, TRAJ, modes=MODES)
name = lambda m: "NOR" if m is None else f"bin {m}"
print(f"{'mode':9s}" + "".join(f"{k:>12s}" for k in kinds) + f"{'exact(opt)':>12s}")
for m in MODES:
    row = [np.nanmean(IM[m]["wall"][TRAJ["kind"] == k]) for k in kinds]
    print(f"{name(m):9s}" + "".join(f"{v:12.3f}" for v in row) + f"{np.nanmean(IM[m]['wall_exact'][TRAJ['kind'] == 'optimal']):12.3f}")
print("\nwall mass on optimal trajectories by start distance (model | exact):")
print(f"{'mode':9s}" + "".join(f"{b:>16s}" for b in BUCKETS))
opt = TRAJ["kind"] == "optimal"
for m in MODES:
    print(f"{name(m):9s}" + "".join(f"{np.nanmean(IM[m]['wall'][opt & (BUCKET == b)]):7.3f} |{np.nanmean(IM[m]['wall_exact'][opt & (BUCKET == b)]):7.3f}" for b in BUCKETS))
print("\ndynamics head, mass off the real next cell (NOR):")
for k in kinds:
    dw = IM[None]["dyn_wrong"][TRAJ["kind"] == k]
    print(f"  {k:10s} mean {np.nanmean(dw):.4f}   steps > 0.5: {np.nanmean(dw > 0.5):.4f}")


In [ ]:
# wall mass along optimal trajectories vs remaining distance, per mode, against the exact conditioned walk
fig, ax = plt.subplots(figsize=(7, 4))
rem = RP["remaining"][:, :MAZE.T]
for m in MODES:
    w, we = IM[m]["wall"][opt], IM[m]["wall_exact"][opt]
    r = rem[opt]
    rs = np.arange(1, int(np.nanmax(r)) + 1)
    line, = ax.plot(rs, [np.nanmean(w[r == x]) for x in rs], marker=".", label=name(m))
    ax.plot(rs, [np.nanmean(we[r == x]) for x in rs], ls="--", color=line.get_color(), lw=1)
ax.set_xlabel("remaining distance (optimal trajectories)"); ax.set_ylabel("policy mass on wall moves (solid: model, dashed: exact)")
ax.grid(alpha=.3); ax.legend(fontsize=7); fig.tight_layout(); plt.show()


## 4. Conditioned rollouts in the real maze

The policy acts in the real maze from every start cell, conditioned on an **attainable** bin: the start's best bin (offset 0) and the next slower non-empty bins (offsets 1, 2). Scored on what was actually achieved. References: NOR rollouts (the model's unconditioned behaviour) and the exact random walk's odds of achieving at least the requested bin from that start. The exact conditioned walk achieves its request with probability 1, so that is the ceiling.

In [ ]:
from maze_consistency.evaluate import make_action_logits
AL = make_action_logits(MODEL, TOK)
N_PER_START, OFFSETS, GREEDY = 4, (0, 1, 2), False
RO = P.conditioned_rollouts(params, AL, TOK, MAZE, GT, seed=0, n_per_start=N_PER_START, offsets=OFFSETS, greedy=GREEDY)
RB = P.dist_bucket(RO["start_dist"]); RBUCKETS = sorted(set(RB), key=lambda b: int(b.split('-')[0]))
lab = lambda o: "NOR" if o < 0 else f"best-{o}" if o else "best"
print(f"{len(RO['start'])} rollouts, {'greedy' if GREEDY else 'sampled'} actions\n")
print(f"{'request':8s}{'n':>6s}{'reached':>9s}{'got req':>9s}{'rw odds':>9s}{'norm R':>9s}{'mean bin':>10s}{'req bin':>9s}")
for o in (-1, *OFFSETS):
    m = RO["offset"] == o
    if not m.any(): continue
    gr = np.nanmean(RO["got_request"][m]) if o >= 0 else np.nan
    rw = np.nanmean(RO["p_request_rw"][m]) if o >= 0 else np.nan
    print(f"{lab(o):8s}{m.sum():6d}{RO['reached'][m].mean():9.2f}{gr:9.2f}{rw:9.3f}{RO['norm_return'][m].mean():9.3f}{RO['achieved'][m].mean():10.2f}{np.nanmean(np.where(RO['requested'][m] >= 0, RO['requested'][m], np.nan)):9.2f}")
print("\ngot request by start distance (model | exact random walk):")
print(f"{'request':8s}" + "".join(f"{b:>18s}" for b in RBUCKETS))
for o in OFFSETS:
    m = RO["offset"] == o
    print(f"{lab(o):8s}" + "".join(f"{np.nanmean(RO['got_request'][m & (RB == b)]):8.2f} |{np.nanmean(RO['p_request_rw'][m & (RB == b)]):8.3f}" for b in RBUCKETS))
print("\nreached the goal at all, by start distance:")
print(f"{'request':8s}" + "".join(f"{b:>10s}" for b in RBUCKETS))
for o in (-1, *OFFSETS):
    m = RO["offset"] == o
    print(f"{lab(o):8s}" + "".join(f"{RO['reached'][m & (RB == b)].mean():10.2f}" for b in RBUCKETS))


In [ ]:
# normalized return (R / R_opt) by start distance and request; 1 = optimal from that start
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ds = np.arange(1, int(RO["start_dist"].max()) + 1)
for o in (-1, *OFFSETS):
    m = RO["offset"] == o
    ax[0].plot(ds, [RO["norm_return"][m & (RO["start_dist"] == dd)].mean() if (m & (RO["start_dist"] == dd)).any() else np.nan for dd in ds], marker=".", label=lab(o))
    if o >= 0:
        ax[1].plot(ds, [np.nanmean(RO["got_request"][m & (RO["start_dist"] == dd)]) if (m & (RO["start_dist"] == dd)).any() else np.nan for dd in ds], marker=".", label=lab(o))
m0 = RO["offset"] == 0
ax[1].plot(ds, [np.nanmean(RO["p_request_rw"][m0 & (RO["start_dist"] == dd)]) for dd in ds], ls="--", color="k", lw=1, label="random walk, best")
ax[0].set_ylabel("R / R_opt(start)"); ax[1].set_ylabel("P(achieved >= requested)")
for a in ax: a.set_xlabel("start distance"); a.grid(alpha=.3); a.legend(fontsize=7)
fig.tight_layout(); plt.show()
